In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

WITH_ADDUCT_MASS_CSV = "../../testset_predictions_with_adduct_mass.csv"
WITHOUT_ADDUCT_MASS_CSV = "../../testset_predictions_without_adduct_mass.csv"

with_df = pd.read_csv(WITH_ADDUCT_MASS_CSV)
without_df = pd.read_csv(WITHOUT_ADDUCT_MASS_CSV)

# Global style settings
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], # Standard crisp fonts
    "axes.linewidth": 1.5,               # Thicker axis lines
    "axes.spines.top": False,            # Remove top spine
    "axes.spines.right": False,          # Remove right spine
    "xtick.major.width": 1.5,            # Match tick thickness to axis
    "ytick.major.width": 1.5,
    "xtick.direction": "out",            # Ticks point outside
    "ytick.direction": "out",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.labelweight": "normal",
})

print(f"With adduct mass: {len(with_df)} rows | Without adduct mass: {len(without_df)} rows")

# R1-#8: is the adduct-mass feature's improvement statistically significant?

Not a comparison of each model's own `adduct_metrics.csv` summary -- those are two independently-aggregated numbers, not paired per-sample errors. Here the two `testset_predictions.csv`-style files (one from a model trained with `adduct_mass` in `calculate_base_features`, one from the same model retrained with it removed) are merged on `(SMILES, Adduct)` so each row is the *same* test molecule scored by both models, then compared with a paired Wilcoxon signed-rank test on relative error -- the right test here since relative CCS error is skewed/non-negative, not normally distributed, so a paired t-test's normality assumption doesn't hold.

In [ ]:
merged = with_df.merge(
    without_df, on=["SMILES", "Adduct"], suffixes=("_with", "_without"),
)

print(f"Matched rows: {len(merged)} / {len(with_df)} (with) / {len(without_df)} (without)")

ccs_true_mismatch = (merged["CCS_True_with"] - merged["CCS_True_without"]).abs() > 1e-6
if ccs_true_mismatch.any():
    print(f"WARNING: {ccs_true_mismatch.sum()} matched rows have differing CCS_True between files -- "
          f"these two runs may not share the same test_data.csv")

ccs_true = merged["CCS_True_with"]
rel_err_with = (merged["CCS_Pred_with"] - ccs_true).abs() / ccs_true * 100
rel_err_without = (merged["CCS_Pred_without"] - ccs_true).abs() / ccs_true * 100

print(f"With adduct mass    | mean RE: {rel_err_with.mean():.4f}% | median RE: {rel_err_with.median():.4f}%")
print(f"Without adduct mass | mean RE: {rel_err_without.mean():.4f}% | median RE: {rel_err_without.median():.4f}%")

In [ ]:
statistic, p_value = wilcoxon(rel_err_with, rel_err_without)

print(f"Paired Wilcoxon signed-rank test | statistic={statistic:.2f} | p={p_value:.4e}")
print(
    "Significant at alpha=0.05" if p_value < 0.05 else "Not significant at alpha=0.05",
    "-- adduct mass", "reduces" if rel_err_with.median() < rel_err_without.median() else "increases",
    "median relative error",
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

parts = ax.violinplot([rel_err_with, rel_err_without], positions=[0, 1], showmedians=True)
for pc, color in zip(parts["bodies"], ["#2c7bb6", "#d7191c"]):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)

ax.set_xticks([0, 1])
ax.set_xticklabels([f"With adduct mass\n(n={len(merged)})", f"Without adduct mass\n(n={len(merged)})"])
ax.set_ylabel("Relative error (%)")
ax.set_title("Effect of Adduct Mass on Test-Set Relative Error", loc="left", fontweight="bold", pad=15)
ax.text(0, 1.01, f"Wilcoxon p={p_value:.2e}",
        transform=ax.transAxes, fontsize=9, color="gray")

plt.tight_layout()
plt.show()